In [3]:
# ============================================================
# SNN Video Reconstruction — CORRECT TEST SCRIPT
# Uses Strategy3Autoencoder (exact architecture from training)
# ============================================================

# ✏️ CONFIG — Edit these 3 paths only
VIDEO_PATH    = "/kaggle/input/datasets/parhitparivar/wafwacfa/vid1.mp4"
WEIGHTS_PATH  = "/kaggle/input/models/abhirooppamula/hysnn/pytorch/default/1/best_model (1).pth"
OUTPUT_DIR    = "test_output"
NUM_FRAMES    = 17   # how many PNGs to save

# ─────────────────────────────────────────────────────────────
import os, time, warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import numpy as np
from pathlib import Path
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────
# EXACT Architecture from rasp4.ipynb (Strategy3Autoencoder)
# ─────────────────────────────────────────────────────────────

class SurrogateGradient(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return input.gt(0).float()
    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        temp = 10.0
        return grad_output / (temp * torch.abs(input) + 1.0) ** 2

spike_fn = SurrogateGradient.apply

class LIFNeuron(nn.Module):
    def __init__(self, tau=3.0, threshold=0.5, soft_reset=True):
        super().__init__()
        self.tau = tau
        self.threshold = threshold
        self.soft_reset = soft_reset
        self.membrane = None

    def forward(self, x):
        if self.membrane is None:
            self.membrane = torch.zeros_like(x)
        new_membrane = self.membrane * (1 - 1 / self.tau) + x
        spikes = spike_fn(new_membrane - self.threshold)
        if self.soft_reset:
            self.membrane = new_membrane - spikes * self.threshold
        else:
            self.membrane = new_membrane * (1 - spikes)
        return spikes

    def reset(self):
        self.membrane = None

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, stride=1, pad=1):
        super().__init__()
        if in_ch < 4:
            self.conv = nn.Conv2d(in_ch, out_ch, k, stride=stride, padding=pad, bias=False)
            self.bn = nn.BatchNorm2d(out_ch)
            self.dw = False
        else:
            self.depthwise = nn.Conv2d(in_ch, in_ch, k, stride=stride, padding=pad, groups=in_ch, bias=False)
            self.pointwise = nn.Conv2d(in_ch, out_ch, 1, bias=False)
            self.bn = nn.BatchNorm2d(out_ch)
            self.dw = True

    def forward(self, x):
        x = self.depthwise(x) if self.dw else self.conv(x)
        if self.dw: x = self.pointwise(x)
        return self.bn(x)

class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        rd = max(1, ch // r)
        self.fc = nn.Sequential(nn.Linear(ch, rd, bias=False), nn.ReLU(True),
                                nn.Linear(rd, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        a = self.fc(self.avg(x).view(b, c))
        m = self.fc(self.max(x).view(b, c))
        return x * (a + m).view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, 1, keepdim=True)
        mx, _ = torch.max(x, 1, keepdim=True)
        return x * self.sig(self.conv(torch.cat([avg, mx], 1)))

class EfficientResidualBlock(nn.Module):
    def __init__(self, ch, spiking=True):
        super().__init__()
        self.c1 = DepthwiseSeparableConv(ch, ch)
        self.a1 = LIFNeuron() if spiking else nn.ReLU(False)
        self.c2 = DepthwiseSeparableConv(ch, ch)
        self.a2 = LIFNeuron() if spiking else nn.ReLU(False)
        self.att = ChannelAttention(ch, 8)
        self.spiking = spiking

    def forward(self, x):
        out = self.a1(self.c1(x))
        out = self.a2(self.c2(out) + x)
        return self.att(out)

    def reset(self):
        if self.spiking:
            for a in [self.a1, self.a2]:
                if hasattr(a, 'reset'): a.reset()

class CompactDownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, spiking=True):
        super().__init__()
        self.down = DepthwiseSeparableConv(in_ch, out_ch, stride=2)
        self.act = LIFNeuron() if spiking else nn.ReLU(False)
        self.r1 = EfficientResidualBlock(out_ch, spiking)
        self.r2 = EfficientResidualBlock(out_ch, spiking)
        self.sa = SpatialAttention()
        self.spiking = spiking

    def forward(self, x):
        return self.sa(self.r2(self.r1(self.act(self.down(x)))))

    def reset(self):
        if self.spiking and hasattr(self.act, 'reset'): self.act.reset()
        self.r1.reset(); self.r2.reset()

class CompactUpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, skip_ch, spiking=True):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch//2, 4, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch//2)
        self.au = LIFNeuron() if spiking else nn.ReLU(False)
        self.comb = nn.Conv2d(in_ch//2 + skip_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.ac = LIFNeuron() if spiking else nn.ReLU(False)
        self.r1 = EfficientResidualBlock(out_ch, spiking)
        self.r2 = EfficientResidualBlock(out_ch, spiking)
        self.spiking = spiking

    def forward(self, x, skip):
        x = self.au(self.bn1(self.up(x)))
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = self.ac(self.bn2(self.comb(torch.cat([x, skip], 1))))
        return self.r2(self.r1(x))

    def reset(self):
        if self.spiking:
            for a in [self.au, self.ac]:
                if hasattr(a, 'reset'): a.reset()
        self.r1.reset(); self.r2.reset()

class DownsampledSkipConnection(nn.Module):
    def __init__(self, ch, target=(8,8), comp_ch=8):
        super().__init__()
        self.target = target
        self.compress = nn.Conv2d(ch, comp_ch, 1, bias=False)
        self.bn_c = nn.BatchNorm2d(comp_ch)
        self.decompress = nn.Conv2d(comp_ch, ch, 1, bias=False)
        self.bn_d = nn.BatchNorm2d(ch)
        self.gate = nn.Sequential(
            nn.Conv2d(ch * 2, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(False),
            nn.Conv2d(ch, ch, 1, bias=False), nn.BatchNorm2d(ch), nn.Sigmoid()
        )
        self.gate_scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        orig = x.shape[2:]
        xs = F.adaptive_avg_pool2d(x, self.target) if orig != self.target else x
        comp = self.bn_c(self.compress(xs))
        dec = self.bn_d(self.decompress(comp))
        if orig != self.target:
            dec = F.interpolate(dec, size=orig, mode='bilinear', align_corners=False)
        gate = self.gate(torch.cat([dec, x], 1))
        refined = dec + torch.tanh(self.gate_scale) * gate * (x - dec)
        return comp, refined   # ← returns TUPLE (comp, refined)

class Strategy3Encoder(nn.Module):
    def __init__(self, in_ch=3, time_steps=2):
        super().__init__()
        self.T = time_steps
        self.ic = nn.Conv2d(in_ch, 32, 3, padding=1, bias=False)
        self.ibn = nn.BatchNorm2d(32)
        self.ia = nn.ReLU(False)
        self.d1 = CompactDownBlock(32, 48, spiking=False)
        self.d2 = CompactDownBlock(48, 64, spiking=True)
        self.d3 = CompactDownBlock(64, 48, spiking=True)
        self.d4 = CompactDownBlock(48, 32, spiking=True)
        self.d5 = CompactDownBlock(32, 24, spiking=True)
        self.bn = nn.Sequential(
            DepthwiseSeparableConv(24, 16), nn.ReLU(False),
            EfficientResidualBlock(16, True), EfficientResidualBlock(16, True)
        )
        self.sk1 = DownsampledSkipConnection(32)
        self.sk2 = DownsampledSkipConnection(48)
        self.sk3 = DownsampledSkipConnection(64)
        self.sk4 = DownsampledSkipConnection(48)
        self.sk5 = DownsampledSkipConnection(32)

    def forward(self, x):
        x = self.ia(self.ibn(self.ic(x)))
        enc = sd = sc = None
        for _ in range(self.T):
            s1=x;  x1=self.d1(s1)
            s2=x1; x2=self.d2(s2)
            s3=x2; x3=self.d3(s3)
            s4=x3; x4=self.d4(s4)
            s5=x4; x5=self.d5(s5)
            b = self.bn(x5)
            c1,d1=self.sk1(s1); c2,d2=self.sk2(s2); c3,d3=self.sk3(s3)
            c4,d4=self.sk4(s4); c5,d5=self.sk5(s5)
            if enc is None:
                enc=[b]; sc=[[c1,c2,c3,c4,c5]]; sd=[[d1,d2,d3,d4,d5]]
            else:
                enc.append(b)
                sc.append([c1,c2,c3,c4,c5]); sd.append([d1,d2,d3,d4,d5])
        T=self.T
        enc_out = sum(enc)/T
        sd_out = [sum(sd[t][i] for t in range(T))/T for i in range(5)]
        sc_out = [sum(sc[t][i] for t in range(T))/T for i in range(5)]
        return enc_out, sd_out, sc_out

    def reset(self):
        for blk in [self.d1,self.d2,self.d3,self.d4,self.d5]: blk.reset()
        for m in self.bn:
            if hasattr(m,'reset'): m.reset()

class Strategy3Decoder(nn.Module):
    def __init__(self, out_ch=3, time_steps=2):
        super().__init__()
        self.T = time_steps
        self.u1 = CompactUpBlock(16, 24, 32, spiking=True)
        self.u2 = CompactUpBlock(24, 32, 48, spiking=True)
        self.u3 = CompactUpBlock(32, 48, 64, spiking=True)
        self.u4 = CompactUpBlock(48, 64, 48, spiking=True)
        self.u5 = CompactUpBlock(64, 48, 32, spiking=False)
        self.rf = nn.Sequential(
            DepthwiseSeparableConv(48, 32), nn.ReLU(False),
            nn.Conv2d(32, out_ch, 1, bias=True)
        )

    def forward(self, enc, skips):
        recon = None
        for _ in range(self.T):
            x = self.u1(enc, skips[4])
            x = self.u2(x, skips[3])
            x = self.u3(x, skips[2])
            x = self.u4(x, skips[1])
            x = self.u5(x, skips[0])
            x = self.rf(x)
            recon = x if recon is None else recon + x
        return torch.sigmoid(recon / self.T)

    def reset(self):
        for blk in [self.u1,self.u2,self.u3,self.u4,self.u5]: blk.reset()

class Strategy3Autoencoder(nn.Module):
    def __init__(self, in_ch=3, time_steps=2):
        super().__init__()
        self.encoder = Strategy3Encoder(in_ch, time_steps)
        self.decoder = Strategy3Decoder(in_ch, time_steps)

    def forward(self, x):
        enc, sd, sc = self.encoder(x)
        return self.decoder(enc, sd), enc, sc   # returns (recon, enc, sc)

    def reset(self):
        self.encoder.reset(); self.decoder.reset()

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────
RES = 256

def preprocess(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (RES, RES), interpolation=cv2.INTER_AREA)
    t = torch.from_numpy(rgb.astype(np.float32) / 255.0)
    return t.permute(2, 0, 1).unsqueeze(0)

def to_numpy(tensor):
    img = tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)

def compute_metrics(orig_t, recon_t):
    o = to_numpy(orig_t).astype(np.float32) / 255.0
    r = to_numpy(recon_t).astype(np.float32) / 255.0
    p = psnr_fn(o, r, data_range=1.0)
    s = ssim_fn(o, r, data_range=1.0, channel_axis=2)
    return p, s

def save_comparison_png(orig_t, recon_t, path, idx, p, s):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f"Frame {idx:02d} | PSNR: {p:.2f} dB | SSIM: {s:.4f}", fontsize=13, fontweight='bold')
    ax[0].imshow(to_numpy(orig_t)); ax[0].set_title("Original"); ax[0].axis('off')
    ax[1].imshow(to_numpy(recon_t)); ax[1].set_title("Reconstructed"); ax[1].axis('off')
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close()

# ─────────────────────────────────────────────────────────────
# Main Test Function
# ─────────────────────────────────────────────────────────────
def run_test():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"\n{'='*60}")
    print(f"  SNN Video Reconstruction — Test (Strategy3Autoencoder)")
    print(f"{'='*60}")
    print(f"  Device  : {device}")
    print(f"  Video   : {VIDEO_PATH}")
    print(f"  Weights : {WEIGHTS_PATH}")
    print(f"{'='*60}\n")

    out_dir = Path(OUTPUT_DIR)
    frames_dir = out_dir / 'reconstructed_frames'
    out_dir.mkdir(parents=True, exist_ok=True)
    frames_dir.mkdir(exist_ok=True)

    # ── Load model with CORRECT architecture ──
    print("Loading model...")
    model = Strategy3Autoencoder(in_ch=3, time_steps=2).to(device)
    ckpt = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
    # Handle both raw state_dict and wrapped checkpoints
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        state_dict = ckpt['model_state_dict']
    elif isinstance(ckpt, dict) and not any(k.startswith('encoder.') or k.startswith('decoder.') for k in list(ckpt.keys())[:3]):
        state_dict = ckpt.get('state_dict', ckpt)
    else:
        state_dict = ckpt
    model.load_state_dict(state_dict, strict=True)  # strict=True to catch any mismatch
    model.eval()
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  ✅ Weights loaded! ({total_params/1e6:.3f}M params)\n")

    # ── Open video ──
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise FileNotFoundError(f"❌ Cannot open video: {VIDEO_PATH}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    print(f"  Video: {total_frames} frames @ {src_fps:.1f} FPS\n")

    # ── Video writer ──
    vid_path = str(out_dir / 'comparison_video.mp4')
    writer = cv2.VideoWriter(vid_path, cv2.VideoWriter_fourcc(*'mp4v'), src_fps, (RES * 2, RES))

    psnr_list, ssim_list, lat_list = [], [], []
    frame_idx = 0
    model.reset()   # ✅ Correct reset method for Strategy3Autoencoder

    print(f"{'─'*62}")
    print(f"  {'Frame':>6}  {'PSNR (dB)':>10}  {'SSIM':>8}  {'Note'}")
    print(f"{'─'*62}")

    with torch.no_grad():
        while True:
            ret, bgr = cap.read()
            if not ret:
                break
            frame_idx += 1
            inp = preprocess(bgr).to(device)

            t0 = time.perf_counter()
            recon, _, _ = model(inp)   # ✅ Unpack all 3 return values
            lat_list.append((time.perf_counter() - t0) * 1000)

            p, s = compute_metrics(inp, recon)
            psnr_list.append(p); ssim_list.append(s)

            note = ""
            if frame_idx <= NUM_FRAMES:
                png_path = frames_dir / f"frame_{frame_idx:03d}.png"
                save_comparison_png(inp, recon, str(png_path), frame_idx, p, s)
                note = f"→ saved frame_{frame_idx:03d}.png"

            print(f"  {frame_idx:>6}  {p:>10.2f}  {s:>8.4f}  {note}")

            # side-by-side video
            orig_bgr  = cv2.cvtColor(to_numpy(inp),   cv2.COLOR_RGB2BGR)
            recon_bgr = cv2.cvtColor(to_numpy(recon), cv2.COLOR_RGB2BGR)
            cv2.putText(orig_bgr,  "ORIGINAL",       (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0),   2)
            cv2.putText(recon_bgr, "RECONSTRUCTED",  (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
            cv2.putText(recon_bgr, f"PSNR:{p:.1f}dB SSIM:{s:.3f}", (10,60), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,200,0), 1)
            writer.write(np.concatenate([orig_bgr, recon_bgr], axis=1))

    cap.release(); writer.release()

    # ── Summary ──
    print(f"{'─'*62}")
    print(f"\n{'='*60}")
    print(f"  RESULTS ({frame_idx} frames processed)")
    print(f"{'='*60}")
    print(f"  Avg PSNR : {np.mean(psnr_list):.4f} dB")
    print(f"  Avg SSIM : {np.mean(ssim_list):.4f}")
    print(f"  Min PSNR : {np.min(psnr_list):.4f} dB  (frame {np.argmin(psnr_list)+1})")
    print(f"  Max PSNR : {np.max(psnr_list):.4f} dB  (frame {np.argmax(psnr_list)+1})")
    print(f"  Avg Latency : {np.mean(lat_list):.2f} ms/frame")
    print(f"  Throughput  : {1000/np.mean(lat_list):.1f} FPS")
    print(f"{'='*60}\n")

    # ── Metrics chart ──
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    xs = range(1, frame_idx + 1)
    ax1.plot(xs, psnr_list, color='steelblue', lw=1.5)
    ax1.axhline(np.mean(psnr_list), color='red', ls='--', label=f'Avg {np.mean(psnr_list):.2f} dB')
    ax1.set_ylabel('PSNR (dB)', fontsize=11); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax1.set_title('Per-Frame Metrics', fontsize=13)
    ax2.plot(xs, ssim_list, color='darkorange', lw=1.5)
    ax2.axhline(np.mean(ssim_list), color='red', ls='--', label=f'Avg {np.mean(ssim_list):.4f}')
    ax2.set_ylabel('SSIM', fontsize=11); ax2.set_xlabel('Frame', fontsize=11)
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(out_dir / 'metrics_chart.png'), dpi=120, bbox_inches='tight')
    plt.close()

    print(f"  📁 Output folder  : {Path(OUTPUT_DIR).resolve()}")
    print(f"  🖼  Frame PNGs     : {frames_dir}/")
    print(f"  🎬 Comparison video: {vid_path}")
    print(f"  📊 Metrics chart  : {out_dir/'metrics_chart.png'}\n")

run_test()



  SNN Video Reconstruction — Test (Strategy3Autoencoder)
  Device  : cuda
  Video   : /kaggle/input/datasets/parhitparivar/wafwacfa/vid1.mp4
  Weights : /kaggle/input/models/abhirooppamula/hysnn/pytorch/default/1/best_model (1).pth

Loading model...
  ✅ Weights loaded! (0.536M params)

  Video: 240 frames @ 30.0 FPS

──────────────────────────────────────────────────────────────
   Frame   PSNR (dB)      SSIM  Note
──────────────────────────────────────────────────────────────
       1       36.05    0.9980  → saved frame_001.png
       2       35.65    0.9978  → saved frame_002.png
       3       35.62    0.9978  → saved frame_003.png
       4       35.68    0.9978  → saved frame_004.png
       5       35.63    0.9978  → saved frame_005.png
       6       35.70    0.9978  → saved frame_006.png
       7       35.73    0.9978  → saved frame_007.png
       8       35.78    0.9978  → saved frame_008.png
       9       35.71    0.9978  → saved frame_009.png
      10       35.74    0.9978 